# Development-only ensemble selection
Combine saved validation prediction exports from task-compatible models. Class order and sample order are checked before averaging. No DDI output belongs in this notebook.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
from src.evaluation.calibration import fit_temperature, select_binary_threshold
from src.evaluation.ensemble import EnsembleMember, equal_weight_ensemble, fit_validation_weights, apply_fitted_ensemble
from src.evaluation.evaluator import freeze_final_configuration
from src.evaluation.metrics import classification_metrics

config = yaml.safe_load(Path('configs/ensemble.yaml').read_text())
PREDICTION_FILES = []  # Validation CSV exports from 90_general_model_testing.ipynb.
TEST_PREDICTION_FILES = []  # Optional aligned internal development-test exports; never used for fitting.
assert config['prediction_split'] == 'validation'
assert PREDICTION_FILES, 'Add at least one development validation prediction export.'

In [ ]:
frames = [pd.read_csv(path) for path in PREDICTION_FILES]
reference = frames[0]
class_order = tuple(config['class_order'])
reference_labels = tuple(reference.true_label.astype(str))
members = []
for path, frame in zip(PREDICTION_FILES, frames):
    assert tuple(frame.true_label.astype(str)) == reference_labels, 'Validation labels/order differ across members.'
    probabilities = np.array([[json.loads(value)[name] for name in class_order] for value in frame.class_probabilities])
    member_name = str(frame.checkpoint_id.dropna().iloc[0]) if frame.checkpoint_id.notna().any() else Path(path).stem
    members.append(EnsembleMember(member_name, config['task'], class_order, probabilities, tuple(frame.sample_id.astype(str))))
targets = np.array([class_order.index(str(value)) for value in reference.true_label])
equal_probabilities, equal_definition = equal_weight_ensemble(members, missing_member_policy=config['missing_member_policy'])
weighted_definition = fit_validation_weights(members, targets, split='validation', iterations=config['weighted_method']['iterations'], learning_rate=config['weighted_method']['learning_rate'])
weighted_probabilities, _ = apply_fitted_ensemble(members, weighted_definition)
equal_metrics = classification_metrics(targets, probabilities=equal_probabilities, labels=range(len(class_order)), class_names=class_order)
weighted_metrics = classification_metrics(targets, probabilities=weighted_probabilities, labels=range(len(class_order)), class_names=class_order)
pd.DataFrame([equal_metrics, weighted_metrics], index=['equal', 'validation_weighted'])[['macro_f1','balanced_accuracy','roc_auc','pr_auc','brier_score']]

In [ ]:
selected_name, selected_probabilities, selected_definition = max([('equal', equal_probabilities, equal_definition), ('validation_weighted', weighted_probabilities, weighted_definition)], key=lambda item: classification_metrics(targets, probabilities=item[1], labels=range(len(class_order)))['macro_f1'])
calibration, calibration_comparison = fit_temperature(np.log(np.clip(selected_probabilities, 1e-12, 1.0)), targets, class_order=class_order, split='validation')
use_calibration = calibration_comparison['recommended']
final_validation_probabilities = calibration.apply(selected_probabilities, input_type='probabilities') if use_calibration else selected_probabilities
threshold = select_binary_threshold(targets, final_validation_probabilities[:, 1], split='validation', objective=config['threshold']['objective'], minimum_sensitivity=config['threshold']['minimum_sensitivity'])
artifact = {'selection_split': 'validation', 'selected_method': selected_name, 'ensemble': selected_definition, 'calibration': {**calibration.to_dict(), 'enabled': use_calibration, 'comparison': calibration_comparison}, 'threshold': threshold}
output = Path('results/ensemble'); output.mkdir(parents=True, exist_ok=True)
(output / 'development_ensemble.json').write_text(json.dumps(artifact, indent=2))
artifact

In [ ]:
internal_test_metrics = None
if TEST_PREDICTION_FILES:
    test_frames = [pd.read_csv(path) for path in TEST_PREDICTION_FILES]
    test_reference_labels = tuple(test_frames[0].true_label.astype(str))
    test_members = []
    for path, frame in zip(TEST_PREDICTION_FILES, test_frames):
        assert tuple(frame.true_label.astype(str)) == test_reference_labels, 'Internal-test labels/order differ across members.'
        probabilities = np.array([[json.loads(value)[name] for name in class_order] for value in frame.class_probabilities])
        member_name = str(frame.checkpoint_id.dropna().iloc[0]) if frame.checkpoint_id.notna().any() else Path(path).stem
        test_members.append(EnsembleMember(member_name, config['task'], class_order, probabilities, tuple(frame.sample_id.astype(str))))
    test_probabilities, _ = apply_fitted_ensemble(test_members, selected_definition, missing_member_policy=config['missing_member_policy'])
    test_probabilities = calibration.apply(test_probabilities, input_type='probabilities') if use_calibration else test_probabilities
    test_targets = np.array([class_order.index(str(value)) for value in test_frames[0].true_label])
    test_predictions = (test_probabilities[:, 1] >= threshold['threshold']).astype(int)
    internal_test_metrics = classification_metrics(test_targets, test_predictions, test_probabilities, labels=range(len(class_order)), class_names=class_order)
    artifact['internal_development_test_metrics'] = internal_test_metrics
    (output / 'development_ensemble.json').write_text(json.dumps(artifact, indent=2))
internal_test_metrics

In [ ]:
FREEZE_NOW = False
MODEL_DEFINITIONS = []  # name, strategy, checkpoint, and config for each selected member.
if FREEZE_NOW:
    assert MODEL_DEFINITIONS, 'List the selected development checkpoints before freezing.'
    frozen = {'task': config['task'], 'class_order': list(class_order), 'models': MODEL_DEFINITIONS, 'ensemble': {**selected_definition, 'identifier': 'final', 'missing_member_policy': config['missing_member_policy']}, 'preprocessing': {'image_resolution': 224, 'transform': 'src.data.transforms.build_transforms(training=False)', 'metadata_fields': []}, 'threshold': threshold, 'calibration': artifact['calibration'], 'dataset_mappings': {'development': 'persisted manifest labels', 'DDI': 'predeclared compatible binary malignancy mapping only'}, 'development_datasets': sorted(reference.source_dataset.dropna().unique().tolist()), 'bootstrap': config['bootstrap'], 'selection_evidence': 'results/ensemble/development_ensemble.json'}
    freeze_final_configuration(frozen)